# Agentic AI Research Agent
**Flow:** User Query → Tavily Web Search (5 sources) → LLM Synthesizer → Research Answer

In [1]:
!pip install -q langchain langgraph langchain-openai langchain-tavily tavily-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.8/119.8 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.9/554.9 kB 13.0 MB/s eta 0:00:00


In [ ]:
# ── CELL 1: Imports & API Keys ──────────────────────────────────────────────

import os
from typing import TypedDict

from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_tavily import TavilySearch

from langgraph.graph import StateGraph, START, END

os.environ["TAVILY_API_KEY"]  = "tvly-c"
os.environ["GROQ_API_KEY"]    = "gsk_a6L"

In [21]:
# ── CELL 2: State ────────────────────────────────────────────────────────────
# Three fields travel through the graph:
#   query        — the original user question (set at entry, never changed)
#   web_results  — raw Tavily output, filled by web_search_node
#   final_answer — synthesized markdown answer, filled by synthesizer_node

class ResearchState(TypedDict):
    query:        str
    web_results:  str
    search_query:  str
    final_answer: str

In [22]:
# ── CELL 3: Tools & LLM ──────────────────────────────────────────────────────

# Tavily — returns top-5 web results as structured dicts
tavily = TavilySearch(max_results=5)

# Groq LLaMA via OpenAI-compatible endpoint (free tier)
llm = ChatOpenAI(
    model="llama-3.3-70b-versatile",
    temperature=0.3,
    api_key=os.environ["GROQ_API_KEY"],
    base_url="https://api.groq.com/openai/v1"
)

In [23]:
# ── CELL 4: Node 1 — Web Search ──────────────────────────────────────────────
# Calls Tavily, formats the 5 results into a numbered text block,
# and stores it in state["web_results"].

def web_search_node(state: ResearchState) -> dict:

    raw = tavily.invoke({"query": state["search_query"]})

    raw = raw["results"]   # <-- this line is the fix

    # raw is a list of dicts: {url, content, title, score, ...}

    # Build a clean numbered block for the LLM prompt
    formatted_results = []

    for i, result in enumerate(raw, start=1):

        title   = result.get("title",   "No title")
        url     = result.get("url",     "")
        content = result.get("content", "").strip()

        formatted_results.append(
            f"[{i}] {title}\n"
            f"    URL: {url}\n"
            f"    {content}"
        )

    web_results_text = "\n\n".join(formatted_results)

    print(f"    Retrieved {len(raw)} sources.")

    return {"web_results": web_results_text}

In [24]:
# this is for refressing the chat or data so we can process with new flow
from langchain_core.messages import SystemMessage, HumanMessage

REWRITE_PROMPT = """\
You rewrite user questions into a short, high-signal web search query.
Return ONLY the search query text. No explanation, no quotes.
"""

def query_rewrite_node(state):
    print("[0/2] Rewriting query for search...")

    messages = [
        SystemMessage(content=REWRITE_PROMPT),
        HumanMessage(content=state["query"])
    ]
    response = llm.invoke(messages)
    rewritten = response.content.strip()

    print(f"    Search query: {rewritten}")
    return {"search_query": rewritten}


In [25]:
# ── CELL 5: Node 2 — LLM Synthesizer ─────────────────────────────────────────
# Receives both the original query and the 5 web snippets.
# The system prompt instructs the LLM to blend web data with its
# own parametric knowledge into one well-structured answer.

SYSTEM_PROMPT = """\
You are an expert research assistant.

You will be given:
  1. A user research query.
  2. Five web search results (title, URL, snippet).

Your job:
  - Synthesize the web results with your own knowledge.
  - Write a clear, well-structured, comprehensive answer.
  - Cite sources inline using [1], [2], etc. where relevant.
  - Highlight any conflicting information across sources.
  - End with a "Sources" section listing all URLs.
  - Be factual. Do not hallucinate beyond the provided data.
"""


def synthesizer_node(state: ResearchState) -> dict:

    user_prompt = (
        f"Research Query: {state['query']}\n\n"
        f"Web Search Results:\n\n{state['web_results']}"
    )

    messages = [
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=user_prompt)
    ]

    response = llm.invoke(messages)

    return {"final_answer": response.content}

In [26]:
# ── CELL 6: Build Graph ───────────────────────────────────────────────────────
#
#   START → web_search_node → synthesizer_node → END

workflow = StateGraph(ResearchState)

workflow.add_node("web_search",   web_search_node)
workflow.add_node("synthesizer",  synthesizer_node)
workflow.add_node("query_rewrite", query_rewrite_node)

workflow.add_edge(START,            "query_rewrite")
workflow.add_edge("query_rewrite",  "web_search")
workflow.add_edge("web_search",     "synthesizer")
workflow.add_edge("synthesizer",    END)

agent = workflow.compile()

print("Graph compiled successfully.")

Graph compiled successfully.


In [27]:
# ── CELL 7: Run Function ──────────────────────────────────────────────────────

def research(query: str):
    """
    Run the research agent on a query.
    Prints a formatted answer with sources.
    """

    print(f"\n{'='*60}") # this is for making the output butiful
    print(f" QUERY: {query}")
    print(f"{'='*60}\n")

    result = agent.invoke({
        "query": query, "search_query": "", "web_results": "", "final_answer": ""
    })

    print(f"\n{'─'*60}") # same butilful
    print(" RESEARCH ANSWER")
    print(f"{'─'*60}\n")
    print(result["final_answer"])
    print(f"\n{'='*60}\n")

    return result

In [28]:
# ── CELL 8: Test It ───────────────────────────────────────────────────────────

research("What are the latest developments in quantum computing in 2025?")


 QUERY: What are the latest developments in quantum computing in 2025?

[0/2] Rewriting query for search...
    Search query: quantum computing advancements 2025
    Retrieved 5 sources.

────────────────────────────────────────────────────────────
 RESEARCH ANSWER
────────────────────────────────────────────────────────────

The latest developments in quantum computing in 2025 are marked by significant breakthroughs and advancements in the field. According to [1], six major trends dominate the 2025 quantum computing landscape, including increased experimentation with logical qubits, development of specialized hardware and software, and expanded workforce development tools. These trends are supported by investment capital, government support, and demonstrated technical breakthroughs, creating a robust ecosystem for commercial quantum computing development.

One of the notable breakthroughs in 2025 is the achievement of quantum computational supremacy on a useful, real-world problem by

{'query': 'What are the latest developments in quantum computing in 2025?',
 'web_results': '[1] Quantum Computing Industry Trends 2025: A Year of Breakthrough ...\n    URL: https://www.spinquanta.com/news-detail/quantum-computing-industry-trends-2025-breakthrough-milestones-commercial-transition\n    Six major trends dominate the 2025 quantum computing landscape: increased experimentation with logical qubits as error correction matures; development of specialized hardware and software for specific problem classes rather than universal quantum computing approaches; increased networking of noisy intermediate-scale quantum devices together; additional layers of software abstraction facilitating easier quantum programming; expanded workforce development tools and training programs; and continuous improvement in physical qubit performance through novel materials and fabrication techniques. Investment capital, government support, workforce development initiatives, and demonstrated technical

In [29]:
# ── CELL 9: Try Your Own Query ────────────────────────────────────────────────

research("Impact of AI on software engineering jobs")


 QUERY: Impact of AI on software engineering jobs

[0/2] Rewriting query for search...
    Search query: AI impact on software engineering careers
    Retrieved 5 sources.

────────────────────────────────────────────────────────────
 RESEARCH ANSWER
────────────────────────────────────────────────────────────

The impact of AI on software engineering jobs is a complex and multifaceted topic. According to various sources, AI is transforming the software development industry, driving advancements that will reshape the field [1], [2]. While AI has the potential to automate repetitive tasks, it is unlikely to replace human software engineers entirely [2]. Instead, AI will enable developers to focus on more complex, creative aspects of software design.

The data suggests that junior developers (22-25) have seen a 20% employment drop, while senior engineers have grown 6-9% [3]. Additionally, frontend roles are declining, while machine learning engineer positions have exploded 40% in 2025 [

{'query': 'Impact of AI on software engineering jobs',
 'web_results': "[1] The Impact of AI on Engineering Jobs - Intuit Blog\n    URL: https://www.intuit.com/blog/innovative-thinking/ai-impact-engineering-jobs\n    [Skip to main content](https://www.intuit.com/blog/innovative-thinking/ai-impact-engineering-jobs#content). *   [Share on LinkedIn](https://www.linkedin.com/shareArticle?mini=true&url=https%3A%2F%2Fwww.intuit.com%2Fblog%2Finnovative-thinking%2Fai-impact-engineering-jobs%2F&title=The+Impact+of+AI+on+Engineering+Jobs). *   [Print](https://www.intuit.com/blog/innovative-thinking/ai-impact-engineering-jobs#). If you’re looking to build or advance a career in this space, Intuit offers a range of [software engineering opportunities](https://www.intuit.com/careers/teams/software-engineering/), from backend software engineering to AI scientist roles. [Learning AI](https://www.intuit.com/blog/innovative-thinking/how-to-learn-artificial-intelligence/) and building hands-on experienc